# Wine model diagnostics

Run the command-line pipeline first, then use this notebook to inspect saved predictions, probability calibration, and the records the model gets wrong.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

from src.evaluate import classification_metrics, threshold_search

report_dir = Path("../reports")
notebook_report_dir = report_dir / "notebooks"
notebook_report_dir.mkdir(parents=True, exist_ok=True)


If `reports/test_predictions.csv` is missing, run:

```bash
python -m src.pipeline --tune-threshold
```

In [ ]:
prediction_path = report_dir / "test_predictions.csv"
if not prediction_path.exists():
    raise FileNotFoundError("Run python -m src.pipeline --tune-threshold before this notebook")

predictions = pd.read_csv(prediction_path)
metrics = json.loads((report_dir / "metrics.json").read_text()) if (report_dir / "metrics.json").exists() else {}

{"rows": len(predictions), "columns": list(predictions.columns), "metrics": metrics}


In [ ]:
labels = predictions["good_quality"].to_numpy(dtype=int)
probabilities = predictions["prediction_probability"].to_numpy(dtype=float)

threshold_report = threshold_search(labels, probabilities, metric="f1")
threshold_frame = pd.DataFrame(threshold_report["thresholds"])
threshold_report["best_threshold"], threshold_report["best_score"]


In [ ]:
axis = threshold_frame.plot(
    x="threshold",
    y="score",
    figsize=(7, 4),
    legend=False,
)
axis.axvline(threshold_report["best_threshold"], linestyle="--", color="black")
axis.set_title("F1 across classification thresholds")
axis.set_xlabel("threshold")
axis.set_ylabel("F1")
axis.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(notebook_report_dir / "threshold_diagnostics.png", dpi=160)


In [ ]:
diagnostic_metrics = []
for threshold in [0.35, 0.5, threshold_report["best_threshold"], 0.65]:
    row = classification_metrics(labels, probabilities, float(threshold))
    diagnostic_metrics.append(row)

diagnostic_frame = pd.DataFrame(diagnostic_metrics)
diagnostic_frame[["threshold", "accuracy", "precision", "recall", "f1", "balanced_accuracy"]]


In [ ]:
errors = predictions.loc[~predictions["correct"]].copy()
errors["confidence"] = (errors["prediction_probability"] - 0.5).abs() * 2
error_review = errors.sort_values("confidence", ascending=False).head(12)

error_review[
    [
        "quality",
        "good_quality",
        "prediction_probability",
        "prediction",
        "confidence",
        "alcohol",
        "volatile acidity",
        "sulphates",
    ]
]


In [ ]:
axis = predictions.hist(
    column="prediction_probability",
    by="good_quality",
    bins=20,
    figsize=(8, 4),
    sharex=True,
)
plt.suptitle("Predicted probability by true class")
plt.tight_layout()
plt.savefig(notebook_report_dir / "probability_by_class.png", dpi=160)


In [ ]:
diagnostic_frame.to_csv(notebook_report_dir / "threshold_diagnostics.csv", index=False)
error_review.to_csv(notebook_report_dir / "high_confidence_errors.csv", index=False)
notebook_report_dir
